# Measure a Loss Landscape

Sweep declared parameter directions and distinguish a slice from the full optimization problem.

## Before You Start

Course 11. Entry course: 1. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** dynamics-01_paths.

## What You Will Be Able to Do

- Evaluate a two-direction parameter slice
- Calculate numerical slopes
- Control depth comparisons

## A plot is a slice, not the whole space

A model with millions of parameters has a loss surface in millions of dimensions. A two-dimensional image fixes a center theta and two directions u,v, then evaluates L(theta+a*u+b*v). Changing the direction scale changes the apparent sharpness. Label the center, directions, normalization, data split, and axes. A smooth slice does not prove every direction is smooth.

The coding lab uses a two-layer scalar regression yhat=w2*tanh(w1*x). Given data x,y and a grid of w1,w2, compute mean squared error for each pair. Rows index w1; columns index w2. This deliberately inspectable model supports a real parameter sweep rather than a hand-drawn hill. The finite-difference gradient perturbs each weight by plus/minus epsilon and divides the loss difference by 2*epsilon.

Convert x, y, and both weight grids to floating point before multiplying or squaring. For example, int8 arithmetic turns 64*2 into -128, reversing the tanh response before the loss is measured. Promotion prevents integer wrapping; the resulting real-valued products and losses still need to fit the float64 range.

## Design the depth experiment

For plain versus residual networks, keep the dataset, width, optimizer, learning rate, update count, and random seeds recorded. Equal width at different depth does not imply equal parameter count or compute. Report those differences. Inspect gradients near the input and output before claiming vanishing gradients; poor optimization can have more than one cause.

Layerwise sweeps vary only one layer while holding all other layers fixed. They answer a conditional question, not what training would find after jointly adapting every layer. Randomly rescaled directions and differently normalized layers can manufacture a visual sharpness difference. The experiment therefore reports numbers and axis definitions as well as optional plots.

## Worked Example: A central difference

L(w+0.01)=0.54 and L(w-0.01)=0.50.

1. **Subtract:** The loss difference is 0.04.
2. **Divide:** The parameter separation is 0.02, so the estimated derivative is 2.
3. **Interpret:** A small negative update should reduce this local loss, but a large step may leave the local regime.

Finite differences estimate local sensitivity; they do not establish a global optimum.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Measured slices for yhat=w2*tanh(w1*x)."""
import numpy as np

def landscape(x, y, first_weights, second_weights):
    """Return MSE grid [len(first_weights),len(second_weights)].

Finite nonempty 1D x,y of equal length and finite 1D weight grids assumed.
Rows vary w1, columns vary w2. Average over examples only."""
    raise NotImplementedError('Implement this function, then run the checks.')

def numerical_gradient(loss, theta, epsilon=1e-05):
    """Central-difference gradient of scalar callable loss at 1D theta.

Positive finite epsilon, finite theta assumed. Do not mutate theta."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

L(w+0.001)=1.003 and L(w-0.001)=0.997.

- Central-difference slope

### Worked Answers

- 3: (1.003-.997)/.002=3.

## Experiment

Run `experiments/dynamics_depth.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- The experiment measures actual optimization and gradient norms, not a promised universal residual-network advantage.
- Vary one layer and compare the same fixed data; label direction normalization.

## Transfer Problems

1. Halve epsilon twice and inspect numerical-gradient stability.
2. Compare training and validation slices around the same checkpoint.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[Visualizing the Loss Landscape of Neural Nets](https://arxiv.org/abs/1712.09913). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.


## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Measured slices for yhat=w2*tanh(w1*x)."""
import numpy as np

def landscape(x, y, first_weights, second_weights):
    """Return MSE grid [len(first_weights),len(second_weights)].

Finite nonempty 1D x,y of equal length and finite 1D weight grids assumed.
Rows vary w1, columns vary w2. Average over examples only."""
    raise NotImplementedError('Implement this function, then run the checks.')

def numerical_gradient(loss, theta, epsilon=1e-05):
    """Central-difference gradient of scalar callable loss at 1D theta.

Positive finite epsilon, finite theta assumed. Do not mutate theta."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
x, y = np.array([-2., 0., 1.]), np.array([1., -.2, .7])
first, second = [-1., .2], [-3., .5, 2.]
expected = [[sum((b*np.tanh(a*xi)-yi)**2 for xi, yi in zip(x, y))/len(x) for b in second] for a in first]
assert np.allclose(landscape(x, y, first, second), expected)
for dtype, value, weight in [(np.int8, 2, 64), (np.uint8, 16, 16)]:
    operands = [np.array([values], dtype=dtype) for values in (value, 1, weight, 1)]
    before = [values.copy() for values in operands]
    np.testing.assert_allclose(landscape(*operands), [[0.]], atol=1e-12)
    assert all(np.array_equal(values, saved) for values, saved in zip(operands, before))
theta = np.array([2., -3.])
assert np.allclose(numerical_gradient(lambda z: z[0]**2+3*z[1]**2+z[0]*z[1], theta), [1., -16.], atol=1e-6)
assert np.array_equal(theta, [2, -3])
print('Landscape checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Measured slices for yhat=w2*tanh(w1*x)."""
import numpy as np


def landscape(x, y, first_weights, second_weights):
    """Return MSE grid [len(first_weights),len(second_weights)].

    Finite nonempty 1D x,y of equal length and finite 1D weight grids assumed.
    Rows vary w1, columns vary w2. Average over examples only.
    """
    x, y, first_weights, second_weights = [np.asarray(values, float) for values in (x, y, first_weights, second_weights)]
    predictions = np.tanh(first_weights[:, None, None] * x[None, None, :]) * second_weights[None, :, None]
    return ((predictions - y)**2).mean(axis=-1)


def numerical_gradient(loss, theta, epsilon=1e-5):
    """Central-difference gradient of scalar callable loss at 1D theta.

    Positive finite epsilon, finite theta assumed. Do not mutate theta.
    """
    theta = np.asarray(theta, float)
    gradient = np.empty_like(theta)
    for i in range(len(theta)):
        delta = np.zeros_like(theta); delta[i] = epsilon
        gradient[i] = (loss(theta + delta) - loss(theta - delta)) / (2 * epsilon)
    return gradient


In [ ]:
import numpy as np
x, y = np.array([-2., 0., 1.]), np.array([1., -.2, .7])
first, second = [-1., .2], [-3., .5, 2.]
expected = [[sum((b*np.tanh(a*xi)-yi)**2 for xi, yi in zip(x, y))/len(x) for b in second] for a in first]
assert np.allclose(landscape(x, y, first, second), expected)
for dtype, value, weight in [(np.int8, 2, 64), (np.uint8, 16, 16)]:
    operands = [np.array([values], dtype=dtype) for values in (value, 1, weight, 1)]
    before = [values.copy() for values in operands]
    np.testing.assert_allclose(landscape(*operands), [[0.]], atol=1e-12)
    assert all(np.array_equal(values, saved) for values, saved in zip(operands, before))
theta = np.array([2., -3.])
assert np.allclose(numerical_gradient(lambda z: z[0]**2+3*z[1]**2+z[0]*z[1], theta), [1., -16.], atol=1e-6)
assert np.array_equal(theta, [2, -3])
print('Landscape checks passed.')


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""Measured plain/residual depth sweeps. CPU; three seeds; no promised winner."""
import json
import torch
from torch import nn


def depth_experiment():
    torch.set_num_threads(1)
    reports = []
    for seed in [2, 5, 11]:
        for depth in [2, 8, 16]:
            for residual in [False, True]:
                torch.manual_seed(seed)
                x = torch.randn(160, 8); y = .5*x + .2*torch.sin(x)
                layers = nn.ModuleList([nn.Linear(8, 8, bias=False) for _ in range(depth)])
                for layer in layers:
                    nn.init.normal_(layer.weight, std=.08)
                def forward(inputs):
                    state = inputs
                    for layer in layers:
                        update = torch.tanh(layer(state))
                        state = state + update if residual else update
                    return state
                optimizer = torch.optim.Adam(layers.parameters(), lr=.01)
                initial = float(((forward(x[:120])-y[:120])**2).mean().detach())
                initial_gradients = None
                for step in range(100):
                    optimizer.zero_grad(set_to_none=True)
                    loss = ((forward(x[:120])-y[:120])**2).mean(); loss.backward()
                    if step == 0:
                        initial_gradients = [float(layer.weight.grad.norm()) for layer in layers]
                    optimizer.step()
                with torch.no_grad():
                    train = float(((forward(x[:120])-y[:120])**2).mean())
                    test = float(((forward(x[120:])-y[120:])**2).mean())
                    center = layers[0].weight.clone()
                    direction = torch.randn_like(center); direction /= direction.norm()
                    sweep = []
                    for offset in [-1., -.5, 0., .5, 1.]:
                        layers[0].weight.copy_(center + offset*direction)
                        sweep.append(float(((forward(x[:120])-y[:120])**2).mean()))
                    layers[0].weight.copy_(center)
                reports.append(dict(seed=seed, depth=depth, residual=residual, parameters=depth*64,
                    initial_train_mse=initial, train_mse=train, test_mse=test, initial_layer_gradient_norms=initial_gradients,
                    first_layer_unit_direction_offsets=[-1, -.5, 0, .5, 1], slice_mse=sweep))
    return reports


if __name__ == '__main__':
    print(json.dumps(depth_experiment(), indent=2))
    print('Target is near identity; this favors residual structure. Equal width is not equal compute.')
